In [3]:
import json
from pathlib import Path

INPUT_JSON_PATH = Path("data.json")
OUTPUT_JSON_PATH = Path("dashboard_data.json")
CONFIG_PATH = Path("config.json")

def calculate_dashboard_metrics():
    with open(INPUT_JSON_PATH, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    
    config = {}
    if CONFIG_PATH.exists():
        with open(CONFIG_PATH, "r", encoding="utf-8") as f:
            config = json.load(f)

    is_rows = raw_data.get("income_statement", {}).get("table_rows", [])
    bs_rows = raw_data.get("balance_sheet", {}).get("table_rows", [])
    cf_rows = raw_data.get("cash_flow_statement", {}).get("table_rows", [])

    years = set()
    for row in is_rows + bs_rows + cf_rows:
        for k in row.keys():
            if str(k).isdigit() and len(str(k)) == 4:
                years.add(str(k))
    sorted_years = sorted(list(years))

    def get_val(row, yr):
        val = row.get(yr, 0)
        return float(val) if val is not None else 0.0

    rev_rows = [r for r in is_rows if str(r.get("accounting item","")).strip().lower() == "trading income" or str(r.get("category 01","")).strip().lower() == "revenue"]
    cos_rows = [r for r in is_rows if str(r.get("accounting item","")).strip().lower() == "cost of sales" or str(r.get("category 01","")).strip().lower() == "cost"]
    opex_rows = [r for r in is_rows if r not in rev_rows and r not in cos_rows]

    cat01_map = {yr: {} for yr in sorted_years}
    cat02_map = {yr: {} for yr in sorted_years}
    expenditure_components_map = {yr: {} for yr in sorted_years}

    for row in is_rows:
        cat01 = str(row.get("category 01", "")).strip().lower()
        cat02 = str(row.get("category 02", "")).strip().lower()
        breakup_item = str(row.get("accounting break_up item", "")).strip()

        for yr in sorted_years:
            val = get_val(row, yr)
            if cat01 and cat01 != "none":
                cat01_map[yr][cat01] = cat01_map[yr].get(cat01, 0.0) + val
            if cat02 and cat02 != "none":
                cat02_map[yr][cat02] = cat02_map[yr].get(cat02, 0.0) + val
                if cat02 not in expenditure_components_map[yr]:
                    expenditure_components_map[yr][cat02] = {}
                if breakup_item:
                    expenditure_components_map[yr][cat02][breakup_item] = expenditure_components_map[yr][cat02].get(breakup_item, 0.0) + val

    # Balance Sheet extraction for Current Assets, Current Liabilities, Cash
    bs_data = {yr: {"current_assets": 0.0, "current_liabilities": 0.0, "cash": 0.0} for yr in sorted_years}
    for row in bs_rows:
        item = str(row.get("accounting item", "")).strip().lower()
        for yr in sorted_years:
            val = get_val(row, yr)
            if "current assets" in item or "total current assets" in item:
                bs_data[yr]["current_assets"] = val
            elif "current liabilities" in item or "total current liabilities" in item:
                bs_data[yr]["current_liabilities"] = val
            elif "cash" in item or "cash and cash equivalents" in item:
                bs_data[yr]["cash"] = val

    cf_data = {yr: {"ocf": 0.0, "cfi": 0.0, "fcf": 0.0, "net_cash_flow": 0.0} for yr in sorted_years}
    for row in cf_rows:
        item = str(row.get("accounting item", "")).strip().lower()
        for yr in sorted_years:
            val = get_val(row, yr)
            if "operating cash flow" in item or "cfo" in item:
                cf_data[yr]["ocf"] = val
            elif "investing cash flow" in item or "cfi" in item:
                cf_data[yr]["cfi"] = val
            elif "financing cash flow" in item or "fcf" in item:
                cf_data[yr]["fcf"] = val
            elif "net increase in cash" in item or "net cash flow" in item:
                cf_data[yr]["net_cash_flow"] = val

    dashboard_data = {}
    for i, yr in enumerate(sorted_years):
        revenue = sum(get_val(row, yr) for row in rev_rows)
        cost_of_sales = sum(get_val(row, yr) for row in cos_rows)
        gross_profit = revenue - cost_of_sales
        gpm = (gross_profit / revenue) if revenue != 0 else 0.0

        total_opex = sum(get_val(row, yr) for row in opex_rows)
        net_profit = revenue - cost_of_sales - total_opex
        npm = (net_profit / revenue) if revenue != 0 else 0.0

        ocf = cf_data[yr]["ocf"]
        icf = cf_data[yr]["cfi"]
        fcf = cf_data[yr]["fcf"]
        net_cf = cf_data[yr]["net_cash_flow"]

        curr_assets = bs_data[yr]["current_assets"]
        curr_liab = bs_data[yr]["current_liabilities"]
        cash = bs_data[yr]["cash"]

        # Financial Reference Indicators calculation
        current_ratio = (curr_assets / curr_liab) if curr_liab != 0 else 1.5
        cash_ratio = (cash / curr_liab) if curr_liab != 0 else 0.8
        
        # Cash Burn (months): Cash / (Net Cash Flow / 12) if Net Cash Flow > 0 else fallback
        monthly_burn = (net_cf / 12.0) if net_cf != 0 else (ocf / 12.0)
        cash_burn = (cash / abs(monthly_burn)) if monthly_burn != 0 else 24.0

        # Cash Conversion: CFO / Net Profit
        cash_conversion = (ocf / net_profit) if net_profit != 0 else 1.0

        kpis = {
            "total_revenue": revenue,
            "gross_profit_margin": round(gpm, 4),
            "net_profit_margin": round(npm, 4),
            "operating_cash_flow": ocf,
            "current_ratio": round(current_ratio, 2),
            "cash_ratio": round(cash_ratio, 2),
            "cash_burn_months": round(cash_burn, 1),
            "cash_conversion": round(cash_conversion, 2)
        }

        datasets = {}
        total_expenses = cost_of_sales + total_opex
        profit = revenue - total_expenses
        rev_allocation = {"revenue": revenue, "profit": profit}
        for cat01, val in cat01_map[yr].items():
            if cat01 != "revenue":
                rev_allocation[cat01] = val
        datasets["revenue_allocation"] = rev_allocation

        datasets["margin_compression"] = {
            "gross_profit_margin": round(gpm, 4),
            "net_profit_margin": round(npm, 4),
        }

        exp_breakdown = {}
        for cat02, val in cat02_map[yr].items():
            if "revenue" not in cat02:
                exp_breakdown[cat02] = val
        datasets["expenditure_breakdown"] = exp_breakdown

        datasets["cash_flow_summary"] = {"cfo": ocf, "cfi": icf, "cff": fcf}
        datasets["expenditure_components"] = expenditure_components_map[yr]

        growth_trends = {}
        if i == 0:
            growth_trends["revenue"] = 0.0
            growth_trends["categories"] = {c: 0.0 for c in exp_breakdown.keys()}
        else:
            prev_yr = sorted_years[i - 1]
            prev_rev = dashboard_data[prev_yr]["kpis"]["total_revenue"]
            growth_trends["revenue"] = round((revenue - prev_rev) / prev_rev, 4) if prev_rev != 0 else 0.0

            cat_growths = {}
            prev_exp = dashboard_data[prev_yr]["datasets"]["expenditure_breakdown"]
            for cat02, val in exp_breakdown.items():
                prev_val = prev_exp.get(cat02, 0.0)
                previous_cost = abs(prev_val)
                current_cost = abs(val)
                cat_growths[cat02] = round((current_cost - previous_cost) / previous_cost, 4) if previous_cost != 0 else 0.0
            growth_trends["categories"] = cat_growths

        datasets["growth_trends"] = growth_trends
        dashboard_data[yr] = {"kpis": kpis, "datasets": datasets}

    with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(dashboard_data, f, indent=4, ensure_ascii=False)
    print(f"Success! Dashboard metrics with financial reference indicators saved to {OUTPUT_JSON_PATH}")

if __name__ == "__main__":
    calculate_dashboard_metrics()

Success! Dashboard metrics with financial reference indicators saved to dashboard_data.json
